# Goalkeepers

### Import Libraries

In [1]:
from selenium import webdriver 
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
import pandas as pd
import requests
from lxml import html
from bs4 import BeautifulSoup


### Create web pages variables to display goalkeeper statistics by league

In [2]:
pl = "https://fbref.com/en/comps/9/gca/Premier-League-Stats"
liga = "http://fbref.com/en/comps/12/gca/La-Liga-Stats"
seriea = "https://fbref.com/en/comps/11/gca/Serie-A-Stats"
bundesliga = "https://fbref.com/en/comps/20/gca/Bundesliga-Stats"
ligue1 = "https://fbref.com/en/comps/13/gca/Ligue-1-Stats"


### Create DataFrame

In [3]:
FBREFplayer = pd.DataFrame(columns = [
    'Player', 'Nation', 'Pos', 'Squad', 'Age', 'Born', '90s',
    'SCA', 'SCA90', 
    'SCA_PassLive', 'SCA_PassDead', 'SCA_TO', 'SCA_Sh', 'SCA_Fld', 'SCA_Def',
    'GCA', 'GCA90',
    'GCA_PassLive', 'GCA_PassDead', 'GCA_TO', 'GCA_Sh', 'GCA_Fld', 'GCA_Def'
])

### Defining path variable

In [4]:
brave_path = "C:/Program Files/BraveSoftware/Brave-Browser/Application/brave.exe" 

### playerscraper Function
Scrapes Player statistics from a league webpage URL and inserts the extracted data into a pandas DataFrame

In [5]:
# Create a list to hold infixed rows
infixed_rows = []
positions = ['GK', 'DF', 'MF', 'FW', 'FW,DF', 'FW,MF', 'MF,DF', 'MF,FW']

def playerscraper (url):
    # First l'ets set up the web driver with the Brave browser
    options = Options()
    options.binary_location = brave_path
    driver = webdriver.Chrome(service=Service(), options=options)
    driver.get(url)

    # define a variable to hold the GKs table
    table = driver.find_element(By.XPATH , '//*[@id="stats_gca"]/tbody')

    # Get all rows in the table
    rows = table.find_elements(By.TAG_NAME, 'tr')

    # Remove the header row


    # Loop through each row and extract the data
    for row in rows:
        # transform the row data into a list
        row_data = row.text.split(' ')
        
        # skip the header rows
        if row_data[0] == 'Rk':
            continue
        # Remove the first element (the rank)
        row_data.pop(0)  

        # Remove the 'Matches' element if it exists
        row_data.remove('Matches')
        
        # for players with name of one word
        if row_data[3] in positions:
            # remove the duplicate of nationality            
            row_data.pop(1)
            
            # Fixing the team name
            if len(row_data[4]) > 2 or row_data[4] == '05':
                # If the team name is split, combine it
                team = row_data.pop(4) + ' ' + row_data.pop(4)
                row_data.insert(4, team)

        # for players with name of more than one word
        elif row_data[4] in positions:
            # remove the duplicate of nationality
            row_data.pop(2)

            # Fixing the player name and team
            name = row_data.pop(0) + ' ' + row_data.pop(0)
            row_data.insert(0, name)
            if len(row_data[4]) > 2 or row_data[4] == '05':
                # If the team name is split, combine it
                team = row_data.pop(3) + ' ' + row_data.pop(3)
                row_data.insert(3, team)

        # for players with name of 3 parts
        elif row_data[5] in positions:
            # remove the duplicate of nationality
            row_data.pop(3)
            # Fixing the player name and team
            name = row_data.pop(0) + ' ' + row_data.pop(0)+ ' '+ row_data.pop(0)
            row_data.insert(0, name)
            if len(row_data[6]) > 2 or row_data[6] == '05':
                # If the team name is split, combine it
                team = row_data.pop(3) + ' ' + row_data.pop(3)
                row_data.insert(3, team)
        else :
            # If the player name is not in the expected format, skip the row
            continue

        # Check if the row has the expected number of elements
        if len(row_data) != 23:
            continue
        # insert the row data into the DataFrame
        FBREFplayer.loc[len(FBREFplayer)] = row_data
    # Convert the DataFrame columns to appropriate data types
    FBREFplayer['90s'] = pd.to_numeric(FBREFplayer['90s'], errors='coerce')
    FBREFplayer['SCA'] = pd.to_numeric(FBREFplayer['SCA'], errors='coerce')
    FBREFplayer['SCA90'] = pd.to_numeric(FBREFplayer['SCA90'], errors='coerce')
    FBREFplayer['SCA_PassLive'] = pd.to_numeric(FBREFplayer['SCA_PassLive'], errors='coerce')
    FBREFplayer['SCA_PassDead'] = pd.to_numeric(FBREFplayer['SCA_PassDead'], errors='coerce')
    FBREFplayer['SCA_TO'] = pd.to_numeric(FBREFplayer['SCA_TO'], errors='coerce')
    FBREFplayer['SCA_Sh'] = pd.to_numeric(FBREFplayer['SCA_Sh'], errors='coerce')
    FBREFplayer['SCA_Fld'] = pd.to_numeric(FBREFplayer['SCA_Fld'], errors='coerce')
    FBREFplayer['SCA_Def'] = pd.to_numeric(FBREFplayer['SCA_Def'], errors='coerce')
    FBREFplayer['GCA'] = pd.to_numeric(FBREFplayer['GCA'], errors='coerce')
    FBREFplayer['GCA90'] = pd.to_numeric(FBREFplayer['GCA90'], errors='coerce')
    FBREFplayer['GCA_PassLive'] = pd.to_numeric(FBREFplayer['GCA_PassLive'], errors='coerce')
    FBREFplayer['GCA_PassDead'] = pd.to_numeric(FBREFplayer['GCA_PassDead'], errors='coerce')
    FBREFplayer['GCA_TO'] = pd.to_numeric(FBREFplayer['GCA_TO'], errors='coerce')
    FBREFplayer['GCA_Sh'] = pd.to_numeric(FBREFplayer['GCA_Sh'], errors='coerce')
    FBREFplayer['GCA_Fld'] = pd.to_numeric(FBREFplayer['GCA_Fld'], errors='coerce')
    FBREFplayer['GCA_Def'] = pd.to_numeric(FBREFplayer['GCA_Def'], errors='coerce') 

    # Close the driver after scraping
    driver.quit()
    
    return FBREFplayer

### Scraping phase

In [6]:
playerscraper(pl)
playerscraper(liga)
playerscraper(seriea)
playerscraper(bundesliga)
playerscraper(ligue1)


,Player,Nation,Pos,Squad,Age,Born,90s,SCA,SCA90,SCA_PassLive,...,SCA_Fld,SCA_Def,GCA,GCA90,GCA_PassLive,GCA_PassDead,GCA_TO,GCA_Sh,GCA_Fld,GCA_Def
0,Max Aarons,ENG,DF,Bournemouth,24,2000,1.0,2,2.09,2,...,0,0,0,0.00,0,0,0,0,0,0
1,Joshua Acheampong,ENG,DF,Chelsea,18,2006,1.9,2,1.06,2,...,0,0,0,0.00,0,0,0,0,0,0
2,Tyler Adams,USA,MF,Bournemouth,25,1999,21.8,41,1.88,35,...,1,4,4,0.18,3,0,0,0,0,1
3,Tosin Adarabioyo,ENG,DF,Chelsea,26,1997,15.7,15,0.96,8,...,0,1,1,0.06,0,0,0,1,0,0
4,Simon Adingra,CIV,"FW,MF",Brighton,22,2002,12.2,47,3.86,33,...,0,0,7,0.57,5,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2571,Edon Zhegrova,KVX,"FW,MF",Lille,25,1999,10.9,38,3.50,21,...,2,0,3,0.28,2,0,0,1,0,0
2572,Melvin Zinga,FRA,GK,Angers,22,2002,1.0,0,0.00,0,...,0,0,0,0.00,0,0,0,0,0,0
2573,Luck Zogbé,CIV,DF,Brest,19,2005,7.1,15,2.11,8,...,2,3,1,0.14,1,0,0,0,0,0
2574,Aristide Zossou,CIV,MF,Auxerre,19,2005,0.2,0,0.00,0,...,0,0,0,0.00,0,0,0,0,0,0


### export the DataFrame to a CSV file

In [7]:
FBREFplayer.to_csv('Player - FBREF.csv', index=False)